# Pre Processing and Tokenizing functions

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
def load_calibration_texts(path):
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                texts.append(line)
    return texts

calib_texts = load_calibration_texts(
    "/content/drive/MyDrive/FYP/Datasets/wikitext/train_500.txt"
)

print(len(calib_texts))
print(calib_texts[0])

321
= Robert Boulter =


In [3]:
from transformers import AutoTokenizer

model_name = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)

# Gemma sometimes needs pad fix
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [4]:
import torch

def build_calib_dataloader(texts, tokenizer, seq_len=256):
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=seq_len,
        return_tensors="pt"
    )
    return [(tokenized["input_ids"][i:i+1],)
            for i in range(tokenized["input_ids"].size(0))]



In [5]:
calib_dataloader = build_calib_dataloader(
    calib_texts[:128],
    tokenizer,
    seq_len=256
)

In [6]:
import os

file_path = "/content/drive/MyDrive/FYP/Datasets/wikitext/train_500.txt"
if os.path.exists(file_path):
    print(f"The file {file_path} exists.")
else:
    print(f"The file {file_path} does NOT exist.")

The file /content/drive/MyDrive/FYP/Datasets/wikitext/train_500.txt does NOT exist.


Loading the LLama-7b model


In [7]:
from transformers import AutoModelForCausalLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
)

model.to(device)
model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): GemmaRMSNorm((2048,), 

In [8]:
for i, block in enumerate(model.model.layers):
    print(i, next(block.parameters()).device)


0 cuda:0
1 cuda:0
2 cuda:0
3 cuda:0
4 cuda:0
5 cuda:0
6 cuda:0
7 cuda:0
8 cuda:0
9 cuda:0
10 cuda:0
11 cuda:0
12 cuda:0
13 cuda:0
14 cuda:0
15 cuda:0
16 cuda:0
17 cuda:0


Block layers in cpu and some offloaded

In [9]:
for i, block in enumerate(model.model.layers):
    print(i, next(block.parameters()).device)


0 cuda:0
1 cuda:0
2 cuda:0
3 cuda:0
4 cuda:0
5 cuda:0
6 cuda:0
7 cuda:0
8 cuda:0
9 cuda:0
10 cuda:0
11 cuda:0
12 cuda:0
13 cuda:0
14 cuda:0
15 cuda:0
16 cuda:0
17 cuda:0


attention weights

In [10]:
block = model.model.layers[0]

print(block.self_attn.q_proj)
print(block.self_attn.k_proj)
print(block.self_attn.v_proj)
print(block.self_attn.o_proj)

Linear(in_features=2048, out_features=2048, bias=False)
Linear(in_features=2048, out_features=256, bias=False)
Linear(in_features=2048, out_features=256, bias=False)
Linear(in_features=2048, out_features=2048, bias=False)


act_scales → tells how big activations can get

act_shifts → tells where activations are centered

They are used inside to reduce quantization error



##Scaling and Shifting factors collect

In [11]:
import torch.nn as nn
import functools
from tqdm import tqdm


activation scales

In [12]:
def get_act_scales(model, dataloader, num_samples=128):
    model.eval()
    device = next(model.parameters()).device
    act_scales = {}

    def stat_tensor(name, tensor):
        hidden_dim = tensor.shape[-1]
        tensor = tensor.view(-1, hidden_dim).abs().detach()
        comming_max = torch.max(tensor, dim=0)[0].float().cpu()
        if name in act_scales:
            act_scales[name] = torch.max(act_scales[name], comming_max)
        else:
            act_scales[name] = comming_max

    def stat_input_hook(m, x, y, name):
        if isinstance(x, tuple):
            x = x[0]
        stat_tensor(name, x)

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            hooks.append(
                m.register_forward_hook(
                    functools.partial(stat_input_hook, name=name))
            )

    for i in tqdm(range(num_samples)):
        model(dataloader[i][0].to(device))

    for h in hooks:
        h.remove()

    return act_scales


activation shifts

In [13]:
def get_act_shifts(model, dataloader, num_samples=128):
    model.eval()
    device = next(model.parameters()).device
    act_shifts = {}

    def stat_tensor(name, tensor):
        hidden_dim = tensor.shape[-1]
        tensor = tensor.view(-1, hidden_dim).detach()
        comming_max = torch.max(tensor, dim=0)[0].float().cpu()
        comming_min = torch.min(tensor, dim=0)[0].float().cpu()
        center = (comming_max + comming_min) / 2
        if name in act_shifts:
            act_shifts[name] = 0.99 * act_shifts[name] + 0.01 * center
        else:
            act_shifts[name] = center

    def stat_input_hook(m, x, y, name):
        if isinstance(x, tuple):
            x = x[0]
        stat_tensor(name, x)

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            hooks.append(
                m.register_forward_hook(
                    functools.partial(stat_input_hook, name=name))
            )

    for i in tqdm(range(num_samples)):
        model(dataloader[i][0].to(device))

    for h in hooks:
        h.remove()

    return act_shifts


In [14]:
act_scales = get_act_scales(
    model,
    calib_dataloader,
    num_samples=16
)

act_shifts = get_act_shifts(
    model,
    calib_dataloader,
    num_samples=16
)


100%|██████████| 16/16 [00:01<00:00,  9.41it/s]


In [15]:
import os

BASE = "/content/drive/MyDrive/FYP/quant_stats_gemma"
os.makedirs(f"{BASE}/act_scales", exist_ok=True)
os.makedirs(f"{BASE}/act_shifts", exist_ok=True)

torch.save(act_scales, f"{BASE}/act_scales/gemma-2-2b.pt")
torch.save(act_shifts, f"{BASE}/act_shifts/gemma-2-2b.pt")

print("Saved Gemma act_scales & act_shifts ✅")


Saved Gemma act_scales & act_shifts ✅


###Scale values of some layers